# Naive Bayes (Gaussian) — From Scratch
> Assumes features are conditionally independent given class.

**P(y|X) ∝ P(y) ∏ P(xᵢ|y)**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
np.random.seed(42)

## GaussianNB Class

In [ ]:
class GaussianNB:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.priors, self.means, self.vars = {}, {}, {}
        for c in self.classes:
            Xc = X[y == c]
            self.priors[c] = len(Xc) / len(y)
            self.means[c]  = Xc.mean(axis=0)
            self.vars[c]   = Xc.var(axis=0) + 1e-9   # avoid /0

    def _log_likelihood(self, x, c):
        m, v = self.means[c], self.vars[c]
        return -0.5 * np.sum(np.log(2*np.pi*v) + (x-m)**2/v)

    def predict(self, X):
        preds = []
        for x in X:
            scores = {c: np.log(self.priors[c]) + self._log_likelihood(x, c)
                      for c in self.classes}
            preds.append(max(scores, key=scores.get))
        return np.array(preds)

## Train & Test on Iris

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

nb = GaussianNB()
nb.fit(X_tr, y_tr)
print(f"Accuracy: {accuracy_score(y_te, nb.predict(X_te))*100:.2f}%")

## Visualise Per-Class Gaussians (feature 0)

In [ ]:
feat = 0
xs = np.linspace(X[:,feat].min()-1, X[:,feat].max()+1, 300)
colors = ['steelblue','tomato','seagreen']

plt.figure(figsize=(8,4))
for c, col in zip(nb.classes, colors):
    m, v = nb.means[c][feat], nb.vars[c][feat]
    pdf = (1/np.sqrt(2*np.pi*v)) * np.exp(-0.5*(xs-m)**2/v)
    plt.plot(xs, pdf, color=col, lw=2, label=iris.target_names[c])
    plt.axvline(m, color=col, linestyle='--', alpha=0.5)

plt.xlabel(iris.feature_names[feat]); plt.ylabel("Density")
plt.title(f"Naive Bayes – Per-class Gaussians (feature: {iris.feature_names[feat]})")
plt.legend(); plt.tight_layout(); plt.show()